 # 03_C2.py : C2, centroid seeding (pre-classifier embedding, VAL-refit radii)

 Needs the finished `units_C0` and `fold_splits.json` from 01_C0.py. Z-score preprocessing, stitched evaluation.

 Each round: per-class KMeans on TRAIN sub-window embeddings of the current labeller, radii from class percentiles then re-fitted on

 the fold's VAL embeddings (largest radius still meeting the class target precision), centroid labelling of candidate windows,

 fine-tune FROM THE BASELINE on labelled + pseudo data; the student becomes the next labeller.

 Output: `units_C2/fXX_sY.pkl` + `models/C2/*.pth`. Resumable.

In [ ]:
from thesis_common import *
from sklearn.cluster import KMeans

POOLS = ["A"]                 # "A" (controlled comparison, all seeds); add "B", "AB" for the pool factor (seed 0 only)
init_data(unlab=sorted(set("".join(POOLS))))

C_PCT = {"Other": 90, "Feeding": 90, "Scratching": 90, "Breathing": 80, "Gliding": 85, "Resting": 80, "Swimming": 80}
C_PUR = {"Other": .75, "Feeding": .85, "Scratching": .75, "Breathing": .90, "Gliding": .85, "Resting": .90, "Swimming": .90}
C_MINPTS = {"Other": 15, "Feeding": 15, "Scratching": 15, "Breathing": 40, "Gliding": 15, "Resting": 40, "Swimming": 40}
C_COV = {"Other": .35, "Feeding": .35, "Scratching": .35}   # default .5
C_INFLATE, C_VAL_MIN_PTS, C_VAL_MIN_CORRECT = 3.0, 30, 15


In [ ]:
@torch.no_grad()
def embed(model, X, bs=128):
    """Sub-window (50 step) mean-pooled pre-classifier embedding: (N, W/SUB, 32)."""
    model.eval(); cap = []; h = model.classifier.register_forward_pre_hook(lambda m, i: cap.append(i[0].detach())); Z = []
    try:
        for s in range(0, len(X), bs):
            cap.clear(); model(torch.from_numpy(np.ascontiguousarray(X[s:s + bs])).float().permute(0, 2, 1).to(dev))
            z = cap[0]; B, D, L = z.shape; Z.append(z.reshape(B, D, L // SUB, SUB).mean(-1).permute(0, 2, 1).cpu().numpy())
    finally: h.remove()
    return np.concatenate(Z)

def sub_majority(Y):
    Yw = Y.reshape(len(Y), W // SUB, SUB); cnt = (Yw[..., None] == np.arange(NC)).sum(2)
    uniq = (cnt == cnt.max(2, keepdims=True)).sum(2) == 1; return cnt.argmax(2), uniq

def nearest(Z, C, chunk=50000):
    Zt, Ct = torch.from_numpy(Z), torch.from_numpy(C); nc, nd = [], []
    for s in range(0, len(Zt), chunk):
        d = torch.cdist(Zt[s:s + chunk], Ct); v, i = d.min(1); nc.append(i.numpy()); nd.append(v.numpy())
    return np.concatenate(nc), np.concatenate(nd)

def fit_seeds(model, names_train, names_val, seed, cap=20000):
    Et, Ev = cut_labelled(names_train), cut_labelled(names_val)
    Zt, (yt, ut) = embed(model, Et["X"]), sub_majority(Et["Y"]); Zt, yt = Zt[ut], yt[ut]
    Zv, (yv, uv) = embed(model, Ev["X"]), sub_majority(Ev["Y"]); Zv, yv = Zv[uv], yv[uv]
    rng = np.random.RandomState(seed); cen, ccls, rad, act = [], [], [], []
    sub_all = rng.permutation(len(Zt))[:200000]
    for c, nm in enumerate(CLASS_NAMES):
        Zc = Zt[yt == c]
        if len(Zc) < 5: continue
        if len(Zc) > cap: Zc = Zc[rng.choice(len(Zc), cap, False)]
        k = max(1, min(5, int((yt == c).sum()) // 150)); km = KMeans(k, n_init=5, random_state=seed).fit(Zc)
        for j in range(k):
            ctr = km.cluster_centers_[j].astype(np.float32); d_own = np.linalg.norm(Zc[km.labels_ == j] - ctr, axis=1)
            r = float(np.percentile(d_own, C_PCT[nm])); d_all = np.linalg.norm(Zt[sub_all] - ctr, axis=1); within = d_all <= r
            pur = float((yt[sub_all][within] == c).mean()) if within.any() else 1.0
            ok = pur >= C_PUR[nm] and within.sum() >= C_MINPTS[nm]
            cen.append(ctr); ccls.append(c); rad.append(r); act.append(ok)
    cen, ccls, rad, act = np.stack(cen), np.array(ccls), np.array(rad), np.array(act)
    near, dist = nearest(Zv, cen); train_rad = rad.copy()
    for j in range(len(cen)):
        c = ccls[j]; nm = CLASS_NAMES[c]; tgt = TARGET_PREC.get(nm, TARGET_PREC["default"]); sel = near == j
        if sel.sum() < C_VAL_MIN_PTS: continue
        o = np.argsort(dist[sel]); ds, ok = dist[sel][o], (yv[sel] == c)[o]; nc = np.cumsum(ok); prec = nc / np.arange(1, len(ok) + 1)
        good = (prec >= tgt) & (nc >= C_VAL_MIN_CORRECT)
        if not good.any(): act[j] = False
        else: rad[j] = min(float(ds[np.where(good)[0].max()]), C_INFLATE * train_rad[j]); act[j] = True
    return dict(centers=cen, cls=ccls, radius=rad, active=act, train_radius=train_rad)

def centroid_labels(model, X, seeds):
    Z = embed(model, X); N, S, D = Z.shape; near, dist = nearest(Z.reshape(-1, D), seeds["centers"])
    ok = seeds["active"][near] & (dist <= seeds["radius"][near]); sub = np.where(ok, seeds["cls"][near], -1).reshape(N, S)
    Y = np.repeat(sub, SUB, axis=1); M = Y >= 0; cov = M.mean(1); keep = np.zeros(N, bool)
    for i in np.flatnonzero(M.sum(1) >= MIN_CONF):
        dom = np.bincount(Y[i][M[i]]).argmax(); keep[i] = cov[i] >= C_COV.get(CLASS_NAMES[dom], 0.5)
    return np.where(M, Y, 0).astype(np.int8), M, keep


In [ ]:
def run_c2(pool, ctx):
    fi, seed = ctx["fi"], ctx["seed"]; rows, qrows, srows = [], [], []
    rng = np.random.RandomState(seed * 7 + fi); ent = make_pool(pool); Xc, srcc = draw_windows(ent, N_CAND, rng)
    labeller = ctx["base"]
    base_meta = dict(arm=f"C2|{pool}|natural|pf1.0", cond="C2", pool=pool, sampling="natural", pool_frac=1.0, exp="pool",
                     fold=fi + 1, test_ind=ctx["test_ind"], seed=seed, frac=1.0)
    for r in range(1, PS_ITERS + 1):
        meta = {**base_meta, "round": r}
        sd = fit_seeds(labeller, ctx["names_train"], ctx["names_val"], SEED + fi * 100 + r); lab = lambda X: centroid_labels(labeller, X, sd)
        pickle.dump(sd, open(OUT / "models" / "C2" / f"f{fi+1:02d}_s{seed}_{pool}_r{r}_seeds.pkl", "wb")) if (OUT / "models" / "C2").exists() else None
        for nm, E in (("VAL", ctx["EV"]), ("TEST", ctx["ET"])):
            Y, M, keep = lab(E["X"]); qrows += quality_rows(E["Y"], Y, M, keep, {**meta, "eval_set": nm})
        Yc, Mc, keepc = lab(Xc); sel, Ms, info = select_pseudo(Yc, Mc, keepc, srcc, SEED + fi * 100 + r)
        srows.append({**meta, **info, "n_candidates": len(Xc), "n_keep": int(keepc.sum())})
        if len(sel) == 0: print("no pseudo windows; stopping chain"); break
        ps = PseudoSet(Xc[sel], Yc[sel].astype(np.int64), Ms)
        student = finetune(ctx["base"], ctx["names_train"], 5000 + fi * 100 + r + seed * 7, pseudo=ps)
        save_state(student, "C2", fi, seed, f"{pool}_r{r}")
        row = row_of({**meta, "kind": "pseudo"}, student, ctx); rows.append(row)
        print(f"C2 {pool} round {r}: test macro {row['test_macro']:.3f} | test target {row['test_target']:.3f}")
        labeller = student; del ps
    return finish_chain(rows, ctx, base_meta), qrows, srows, []


In [ ]:
(OUT / "models" / "C2").mkdir(parents=True, exist_ok=True)
run_stage("C2", run_c2, POOLS)
print("C2 units finished:", len(list((OUT / "units_C2").glob("*.pkl"))))
